Hi. First and Test file for OMP.

In [2]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# 1. 초기 조건 셋업 (Curtis Example 2.2 데이터)
m1 = 10**26  # kg
m2 = 10**26  # kg
G = 6.674e-20  # 만유인력 상수 (단위: km^3 / (kg * s^2))

# 물체 1의 초기 위치(km)와 속도(km/s)
R1_0 = np.array([0.0, 0.0, 0.0])
V1_0 = np.array([10.0, 20.0, 30.0])

# 물체 2의 초기 위치(km)와 속도(km/s)
R2_0 = np.array([3000.0, 0.0, 0.0])
V2_0 = np.array([0.0, 40.0, 0.0])

print("✅ 우주 캔버스 초기 셋업 완료! 정상 작동합니다.")

✅ 우주 캔버스 초기 셋업 완료! 정상 작동합니다.


Hi. 
test.

In [1]:
import numpy as np

# 🎯 미션 1: TwoBody 클래스 뼈대 만들기
class TwoBody:
    def __init__(self, mu=398600.0):
        # 기본값으로 지구의 중력 파라미터 세팅 (단위: km^3/s^2)
        self.mu = mu

    # 🎯 미션 2: 테일러 급수(f and g series)를 이용한 위치 예측 함수
    def propagate_by_time_series(self, r0_vec, v0_vec, dt):
        """
        Curtis 교재 식 2.172 기준 4차항까지 전개하여 dt초 후의 위치 벡터를 근사 계산합니다.
        """
        # 1. 초기 벡터의 크기 및 내적 계산
        r0 = np.linalg.norm(r0_vec)
        v0 = np.linalg.norm(v0_vec)
        r0_dot_v0 = np.dot(r0_vec, v0_vec)
        
        # 2. f 계수 계산 (4차항까지)
        f2 = -0.5 * (self.mu / r0**3) * dt**2
        f3 = 0.5 * (self.mu * r0_dot_v0 / r0**5) * dt**3
        f4 = (self.mu / 24.0) * ( (3 * v0**2 / r0**5) - (2 * self.mu / r0**6) - (15 * r0_dot_v0**2 / r0**7) ) * dt**4
        f = 1.0 + f2 + f3 + f4
        
        # 3. g 계수 계산 (4차항까지)
        g3 = -(1.0 / 6.0) * (self.mu / r0**3) * dt**3
        g4 = 0.25 * (self.mu * r0_dot_v0 / r0**5) * dt**4
        g = dt + g3 + g4
        
        # 4. 최종 위치 벡터 계산 (r = f*r0 + g*v0)
        r_vec = f * r0_vec + g * v0_vec
        
        return r_vec, f, g

print("✅ TwoBody 클래스 및 f, g 급수 엔진 장착 완료!")

✅ TwoBody 클래스 및 f, g 급수 엔진 장착 완료!


In [2]:
# 🎯 미션 3: 코드 검증 및 발산 확인 (Example 2.15)

# 1. 지구 환경 객체 생성
earth_system = TwoBody(mu=398600.0)

# 2. 초기 조건 세팅 (근지점 출발 위성)
r0_vec = np.array([7000.0, 0.0, 0.0])
v0_vec = np.array([0.0, 8.2663, 0.0])

# --- 테스트 1: dt = 60초 (1분 후) ---
dt_short = 60.0
r_short, f_short, g_short = earth_system.propagate_by_time_series(r0_vec, v0_vec, dt_short)
print(f"⏱️ 1분 후 예측 위치: {r_short} km")
print(f"📏 1분 후 지구 중심부터의 거리: {np.linalg.norm(r_short):.2f} km\n")

# --- 테스트 2: dt = 600초 (10분 후) ---
dt_long = 600.0
r_long, f_long, g_long = earth_system.propagate_by_time_series(r0_vec, v0_vec, dt_long)
print(f"⏱️ 10분 후 예측 위치: {r_long} km")
print(f"📏 10분 후 지구 중심부터의 거리: {np.linalg.norm(r_long):.2f} km")

⏱️ 1분 후 예측 위치: [6985.36571877  495.63217464    0.        ] km
📏 1분 후 지구 중심부터의 거리: 7002.93 km

⏱️ 10분 후 예측 위치: [5617.43256456 4613.95464       0.        ] km
📏 10분 후 지구 중심부터의 거리: 7269.40 km


In [3]:
import numpy as np
import matplotlib.pyplot as plt

# --- 예제 2.15 초기 조건 셋업 ---
mu_earth = 398600.0  # 지구 중력 파라미터 (km^3/s^2)
r0_vec = np.array([7000.0, 0.0, 0.0])  # 초기 위치 벡터 (km)
v0_vec = np.array([0.0, 8.2663, 0.0])  # 초기 속도 벡터 (km/s)

# 초기 벡터의 크기 및 내적 계산
r0 = np.linalg.norm(r0_vec)
v0 = np.linalg.norm(v0_vec)
r0_dot_v0 = np.dot(r0_vec, v0_vec)  # r0와 v0는 수직이므로 0이 됩니다.

print(f"초기 거리 r0: {r0} km")
print(f"초기 속력 v0: {v0} km/s")
print(f"r0 dot v0: {r0_dot_v0} (수직 확인)")

# --- 시간 설정 및 f, g 계수 사전 계산 ---
time_span = np.linspace(0, 1000, 500)  # 0초부터 1000초까지 500개의 점

# 교재의 Lagrange 계수 f, g 수식 (4차항까지 근사)
# r0_dot_v0 = 0 이므로 f3, g4 항은 사라집니다.
f_coeffs = []
g_coeffs = []

for dt in time_span:
    # f 계수 계산
    f2 = -0.5 * (mu_earth / r0**3) * dt**2
    f4 = (mu_earth / 24.0) * ( (3 * v0**2 / r0**5) - (2 * mu_earth / r0**6) ) * dt**4
    f = 1.0 + f2 + f4
    f_coeffs.append(f)
    
    # g 계수 계산
    g3 = -(1.0 / 6.0) * (mu_earth / r0**3) * dt**3
    g = dt + g3
    g_coeffs.append(g)

f_coeffs = np.array(f_coeffs)
g_coeffs = np.array(g_coeffs)

# --- f and g series를 이용한 거리 r(t) 계산 ---
r_vec_fg = f_coeffs[:, np.newaxis] * r0_vec + g_coeffs[:, np.newaxis] * v0_vec
r_mag_fg = np.linalg.norm(r_vec_fg, axis=1)

print("✅ f and g series 데이터 생성 완료!")

초기 거리 r0: 7000.0 km
초기 속력 v0: 8.2663 km/s
r0 dot v0: 0.0 (수직 확인)
✅ f and g series 데이터 생성 완료!


In [1]:
# --- 수치 적분(numerical integration)을 이용한 엄밀해(Exact) 계산 ---
from scipy.integrate import solve_ivp

# 이체문제 미분 방정식 정의
def two_body_ode(t, y, mu):
    r_vec = y[:3]
    v_vec = y[3:]
    r_mag = np.linalg.norm(r_vec)
    a_vec = -(mu / r_mag**3) * r_vec
    return np.concatenate((v_vec, a_vec))

# 초기 상태 벡터 [r0_x, r0_y, r0_z, v0_x, v0_y, v0_z]
y0 = np.concatenate((r0_vec, v0_vec))

# solve_ivp로 엄밀해 계산 (매우 높은 정확도로 설정)
sol = solve_ivp(two_body_ode, [0, 1000], y0, args=(mu_earth,), t_eval=time_span, rtol=1e-10, atol=1e-10)

# 엄밀해의 거리 계산
r_mag_exact = np.linalg.norm(sol.y[:3], axis=0)

# --- 그래프 시각화 (교재 Figure 2.32 재현) ---
plt.figure(figsize=(10, 6))

# f and g series 결과 (빨간색)
plt.plot(time_span, r_mag_fg, 'r-', label='f and g series (4th order)')

# 엄밀해 결과 (파란색)
plt.plot(time_span, r_mag_exact, 'b-', label='Exact solution')

# 그래프 꾸미기
plt.title('Comparison of Radial Distance: f and g series vs Exact')
plt.xlabel('Time t (sec)')
plt.ylabel('Distance r (km)')
plt.ylim(7000, 7700) # 교재와 같은 y축 범위
plt.grid(True)
plt.legend()

# 발산 시점 표시 (교재 기준 약 10분 = 600초)
plt.axvline(x=600, color='gray', linestyle='--')
plt.text(610, 7100, 'diverge (~10 min)', color='gray')

plt.show() #plot

print("✅ 교재 그래프 재현 완료! 600초(10분) 이후 오차가 폭발하는 것을 확인하세요.")

NameError: name 'np' is not defined